# Parallel Multi-Agent Baseline

This notebook demonstrates the **Parallel Multi-Agent Baseline** - a comparison system that runs all three agents simultaneously on every query, then synthesizes their responses.

## Baseline vs PHA Comparison

| Aspect | PHA (Orchestrated) | Parallel Baseline |
|--------|---------------------|-------------------|
| **Routing** | Intelligent - selects which agents to use | None - always runs all agents |
| **Agent Hierarchy** | Main agent + supporting agents | All agents are peers |
| **Synthesis** | Orchestrated collaboration | Simple LLM synthesis |
| **Efficiency** | Only calls needed agents | Always calls all 3 agents |

This baseline helps demonstrate the value of intelligent routing in the PHA system.

## Setup

First, let's set up the environment and load the necessary modules.

In [ ]:
import os
import sys

# Add parent directory to path for imports
sys.path.insert(0, os.path.abspath('..'))

# Set API keys (replace with your own or set environment variables)
# os.environ['GEMINI_API_KEY'] = 'your-key-here'
# os.environ['TAVILY_API_KEY'] = 'your-key-here'  # Optional, for web search

In [ ]:
from config.settings import Settings
from pha.utils import load_persona
from pha.agents import (
    DataScienceAgent,
    DomainExpertAgent,
    HealthCoachAgent,
    ParallelMultiAgentBaseline,
    create_parallel_baseline,
)

# Load settings and API keys
settings = Settings()
GEMINI_API_KEY = settings.gemini_api_key
TAVILY_API_KEY = settings.tavily_api_key

print(f"Gemini API key configured: {bool(GEMINI_API_KEY)}")
print(f"Tavily API key configured: {bool(TAVILY_API_KEY)}")

## Load Sample Data

Load the sample persona data for the agents to analyze.

In [ ]:
# Load sample persona data
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

print(f"Summary data: {len(summary_df)} days")
print(f"Activities: {len(activities_df)} records")
print(f"Profile: {profile_df['age'].values[0]} year old {profile_df['gender'].values[0]}")

## Option 1: Quick Setup with Factory Function

The easiest way to create a fully configured parallel baseline:

In [ ]:
# Create parallel baseline with factory function
baseline = create_parallel_baseline(
    gemini_api_key=GEMINI_API_KEY,
    tavily_api_key=TAVILY_API_KEY,
    settings=settings,
    debug_verbose=True,
)

print("Parallel baseline created successfully!")

## Option 2: Manual Setup (More Control)

For more control over individual agents, you can configure them separately:

In [ ]:
# # Manual setup (uncomment to use instead of factory function)
# 
# # 1. Create Data Science Agent
# ds_agent = DataScienceAgent(settings=settings)
# ds_agent.configure(gemini_api_key=GEMINI_API_KEY)
# 
# # 2. Create Domain Expert Agent
# de_agent = DomainExpertAgent(tavily_api_key=TAVILY_API_KEY)
# de_agent.get_agent(gemini_api_key=GEMINI_API_KEY)
# 
# # 3. Create Health Coach Agent
# coach_agent = HealthCoachAgent(simple_mode=True)
# coach_agent.configure(gemini_api_key=GEMINI_API_KEY)
# 
# # 4. Create and configure baseline
# baseline = ParallelMultiAgentBaseline(debug_verbose=True)
# baseline.configure(gemini_api_key=GEMINI_API_KEY)
# baseline.set_agents(
#     data_science_agent=ds_agent,
#     domain_expert_agent=de_agent,
#     health_coach_agent=coach_agent,
# )

## Test Query 1: Sleep Analysis

Let's test with a sleep-related query that benefits from all three agents.

In [ ]:
query1 = "How has my sleep been over the past two weeks? Am I getting enough deep sleep?"

print(f"Query: {query1}")
print("=" * 80)
print("\nProcessing... (all 3 agents running in parallel)\n")

response1 = baseline.respond(query1)

print("\n" + "=" * 80)
print("SYNTHESIZED RESPONSE:")
print("=" * 80)
print(response1)

### View Individual Agent Responses

We can also inspect what each agent contributed:

In [ ]:
individual = baseline.get_individual_responses()

print("=" * 80)
print("DATA SCIENCE AGENT RESPONSE:")
print("=" * 80)
print(individual['data_science'][:2000] + "..." if len(individual['data_science']) > 2000 else individual['data_science'])

In [ ]:
print("=" * 80)
print("DOMAIN EXPERT AGENT RESPONSE:")
print("=" * 80)
print(individual['domain_expert'][:2000] + "..." if len(individual['domain_expert']) > 2000 else individual['domain_expert'])

In [ ]:
print("=" * 80)
print("HEALTH COACH AGENT RESPONSE:")
print("=" * 80)
print(individual['health_coach'][:2000] + "..." if len(individual['health_coach']) > 2000 else individual['health_coach'])

## Test Query 2: Heart Health

Let's try a query about cardiovascular health.

In [ ]:
# Reset conversation for a fresh start
baseline.reset_conversation()

query2 = "What does my resting heart rate tell me about my cardiovascular fitness?"

print(f"Query: {query2}")
print("=" * 80)

response2 = baseline.respond(query2)

print("\nSYNTHESIZED RESPONSE:")
print("=" * 80)
print(response2)

## Test Query 3: Activity Recommendations

A query that might benefit more from the Health Coach.

In [ ]:
baseline.reset_conversation()

query3 = "I want to improve my overall fitness. What should I focus on based on my data?"

print(f"Query: {query3}")
print("=" * 80)

response3 = baseline.respond(query3)

print("\nSYNTHESIZED RESPONSE:")
print("=" * 80)
print(response3)

## Comparison Notes

When comparing the Parallel Baseline to PHA:

**Advantages of Parallel Baseline:**
- Simpler architecture
- Always gets input from all perspectives
- No routing errors possible

**Advantages of PHA (Orchestrated):**
- More efficient (doesn't call unnecessary agents)
- Better conversation flow (main agent drives the interaction)
- More coherent responses (orchestrated collaboration vs post-hoc synthesis)
- Can handle multi-turn conversations more naturally

The key research question: **Does intelligent routing add value over brute-force parallelism?**

## Clean Up

In [ ]:
# Reset for next session
baseline.reset_conversation()
print("Conversation reset. Baseline ready for new queries.")